# Component 4 – Tea Maturity Classifier Training

**Function 1:** UAV RGB tile → maturity class (Young / Medium / Mature) + confidence.

Runtime → Change runtime type → **T4 GPU** before running.

Expected in Google Drive: `MyDrive/C4_data/labels.csv` and `MyDrive/C4_data/tiles.zip` (zip of `data/processed/tiles`).

## 1. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# If the repo is private, create a GitHub token (Settings → Developer settings → Fine-grained token,
# read-only, this repo only) and paste it when asked. Never save the token in the notebook.
import getpass, subprocess
REPO = 'https://github.com/ShiharaXcor/AI-Driven-Drone-System-for-Precision-Spraying-in-Sri-Lankan-Tea-Plantations-J26-DS-346.git'
BRANCH = 'component4/maturity-spray-engine'
token = getpass.getpass('GitHub token (leave empty if repo is public): ')
url = REPO.replace('https://', f'https://{token}@') if token else REPO
subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, url, '/content/repo'], check=True)
del token, url
%cd /content/repo/component4_maturity_engine
!git log --oneline -3

In [ ]:
!pip -q install timm scikit-learn

## 2. Copy data from Drive

In [ ]:
DATA = '/content/drive/MyDrive/C4_data'
!mkdir -p data/processed/tiles /content/unz
!cp $DATA/labels.csv data/labels.csv
!unzip -q -o $DATA/tiles.zip -d /content/unz
# works whether the zip holds the tiles directly or inside a folder
!find /content/unz -name '*.jpg' -exec mv -t data/processed/tiles {} +
!ls data/processed/tiles | wc -l

## 3. Check the split (by source photo, stratified)

In [ ]:
from pathlib import Path
from src.classifier.dataset import load_labelled, group_split, describe
items = load_labelled(Path('data/labels.csv'), Path('data/processed/tiles'))
print(describe(group_split(items)))

## 4. Train EfficientNet-B0 (main model)

In [ ]:
!python -m src.classifier.train --labels data/labels.csv --tiles data/processed/tiles \
    --model efficientnet_b0 --epochs 25 --batch 32 --out runs/effnet_b0

## 5. Train MobileNetV3 (comparison model)

In [ ]:
!python -m src.classifier.train --labels data/labels.csv --tiles data/processed/tiles \
    --model mobilenetv3_large_100 --epochs 25 --batch 32 --out runs/mobilenetv3

## 6. Compare models and view results

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
rows = []
for run in ['runs/effnet_b0', 'runs/mobilenetv3']:
    r = json.load(open(f'{run}/results.json'))
    rows.append({'model': r['model'], 'accuracy': r['accuracy'], 'macro_precision': r['macro_precision'],
                 'macro_recall': r['macro_recall'], 'macro_f1': r['macro_f1'],
                 'young_recall': r['per_class']['young']['recall'],
                 'latency_ms_per_tile': r['latency_ms_per_tile']})
    display(Image(f'{run}/confusion_matrix.png'))
    display(Image(f'{run}/training_curves.png'))
pd.DataFrame(rows)

## 7. Save results back to Drive

In [ ]:
!mkdir -p $DATA/runs && cp -r runs/* $DATA/runs/
!ls -R $DATA/runs | head